To perform Recursive Feature Elimination, I need to import the necessary classes: RFE for feature selection and RandomForestClassifier as the estimator. Then, I will initialize both, fit RFE to the training data, and select the top 10 features.

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Initialize RandomForestClassifier as the estimator
estimator = RandomForestClassifier(random_state=42)

# Initialize RFE with the estimator and specify n_features_to_select
n_features_to_select = 10
rfe_selector = RFE(estimator=estimator, n_features_to_select=n_features_to_select)

# Fit RFE to the training data
rfe_selector.fit(X_train, y_train.values.ravel()) # .values.ravel() to convert y_train DataFrame to 1D array

# Get the selected features
selected_features_rfe = X_train.columns[rfe_selector.support_]

# Create new DataFrames with only the selected features
X_train_rfe = X_train[selected_features_rfe]
X_test_rfe = X_test[selected_features_rfe]

print(f"Selected features by RFE ({n_features_to_select} features):")
print(selected_features_rfe.tolist())

print(f"\nShape of X_train_rfe: {X_train_rfe.shape}")
print(f"Shape of X_test_rfe: {X_test_rfe.shape}")

Evaluate the performance of the model trained with features selected by RFE using metrics like classification report and accuracy score on the test set. Additionally, visualize the feature importance determined by the RFE process for better insights. I will initialize a RandomForestClassifier, train it on the RFE-selected training data, make predictions on the test data, and then calculate and display the classification report and accuracy score to evaluate the model's performance.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# Initialize RandomForestClassifier with random_state=42
model = RandomForestClassifier(random_state=42)

# Fit the model to the RFE-selected training data
model.fit(X_train_rfe, y_train.values.ravel())

# Make predictions on the RFE-selected test data
y_pred = model.predict(X_test_rfe)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Print accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f}")

**Reasoning**:
Now that the model has been evaluated, I will extract, sort, and visualize the feature importances from the trained RandomForestClassifier to gain insights into which features contributed most to the model's predictions, as specified in the instructions.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extract feature importances
feature_importances = model.feature_importances_

# Create a Pandas Series for better readability and indexing
feature_importance_series = pd.Series(feature_importances, index=selected_features_rfe)

# Sort the feature importances in descending order
feature_importance_series_sorted = feature_importance_series.sort_values(ascending=False)

print("Feature Importances from RFE-selected RandomForest Model:")
display(feature_importance_series_sorted)

# Create a bar plot to visualize the sorted feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x=feature_importance_series_sorted.values, y=feature_importance_series_sorted.index, palette='viridis', hue=feature_importance_series_sorted.index, legend=False)
plt.title('Feature Importances from RFE-selected RandomForest Model')
plt.xlabel('Feature Importance')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()

**After using RFE Model:**

•	The RFE process effectively reduced dimensionality while maintaining high model performance, suggesting that these 10 features are highly discriminative for attack type classification. This parsimonious feature set can lead to more efficient and interpretable models.

•	Despite high overall accuracy, the lower performance on certain minority classes warrants further investigation. Future steps should involve techniques like re-sampling (e.g., SMOTE) or exploring different model architectures to improve the classification of these specific attack types.


In the subsequent step, we will employ tree-based feature importance for our dataset. Since RandomForestClassifier was utilized during the RFE process and its feature importances were obtained, a method of tree-based embedded feature selection has already been applied. The feature_importances_ attribute from a trained RandomForestClassifier, or similar tree-based models, can be used to select features that exceed a specified threshold or to identify the top N features according to their importance scores.

In [ ]:
n_top_features_tree = 10

# Get the names of the top N features based on feature importance from the RandomForest model
top_features_tree_names = feature_importance_series_sorted.head(n_top_features_tree).index.tolist()

# Update X to include only these selected features
X_tree_selected = X_train[top_features_tree_names] # Using X_train here to align with the training split

print(f"X DataFrame (training set) updated to include only the top {n_top_features_tree} features based on tree importance.")
print("Selected features:")
display(X_tree_selected.head())
print(f"New shape of X_tree_selected: {X_tree_selected.shape}")

# Also create a test set with these selected features
X_test_tree_selected = X_test[top_features_tree_names]
print(f"New shape of X_test_tree_selected: {X_test_tree_selected.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a bar plot to visualize the sorted feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x=feature_importance_series_sorted.values, y=feature_importance_series_sorted.index, palette='viridis', hue=feature_importance_series_sorted.index, legend=False)
plt.title('Feature Importances from RFE-selected RandomForest Model')
plt.xlabel('Feature Importance')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()

For the first model, I’ll be initializing a Logistic Regression model and train it on the train it on the RFE-selected training data, and evaluating its performance on the test set using a classification report and accuracy score.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Initialize Logistic Regression model
# Set solver to 'liblinear' for multiclass classification with L1/L2 regularization
# Set max_iter to a sufficiently large number for convergence
model_lr = LogisticRegression(random_state=42, solver='liblinear', multi_class='auto', max_iter=1000)

# Fit the model to the RFE-selected training data
# .values.ravel() to convert y_train DataFrame to 1D array
model_lr.fit(X_train_rfe, y_train.values.ravel())

# Make predictions on the RFE-selected test data
y_pred_lr = model_lr.predict(X_test_rfe)

# Print classification report
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred_lr))

# Print accuracy score
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy Score for Logistic Regression: {accuracy_lr:.4f}")

Comparing the performance of LogisticRegression to RandomForestClasiffier. The RandomForestClassifer performed better with accuracy of 97.68%.

Initialize a DecisionTreeClassifier model, train it on the RFE-selected training data, and evaluate its performance on the test set using a classification report and accuracy score.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize Decision Tree Classifier model
model_dt = DecisionTreeClassifier(random_state=42)

# Fit the model to the RFE-selected training data
# .values.ravel() to convert y_train DataFrame to 1D array
model_dt.fit(X_train_rfe, y_train.values.ravel())

# Make predictions on the RFE-selected test data
y_pred_dt = model_dt.predict(X_test_rfe)

# Print classification report
print("Classification Report for Decision Tree:")
print(classification_report(y_test, y_pred_dt))

# Print accuracy score
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"Accuracy Score for Decision Tree: {accuracy_dt:.4f}")

Decision Tree outperforms Logistic Regression in accuracy, while Random Forest performs slightly better than Decision Tree.

Since we’ve already used RandomForest, I’ll now initialize an XGBoost Classifier, train it on RFE-selected data, and evaluate its performance with a classification report and accuracy score. XGBoost is a fast, efficient gradient boosting library that solves data science problems with accurate parallel tree boosting algorithms.



In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score

# Initialize XGBoost Classifier model
# For multiclass classification, objective='multi:softmax' is used.
# num_class must be specified for 'multi:softmax'. We have 12 unique classes.
model_xgb = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(y_train['Attack_type'].unique()),
    eval_metric='mlogloss', # Metric for multiclass classification
    random_state=42
)

# Fit the model to the RFE-selected training data
model_xgb.fit(X_train_rfe, y_train.values.ravel())

# Make predictions on the RFE-selected test data
y_pred_xgb = model_xgb.predict(X_test_rfe)

# Print classification report
print("Classification Report for XGBoost:")
print(classification_report(y_test, y_pred_xgb))

# Print accuracy score
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Accuracy Score for XGBoost: {accuracy_xgb:.4f}")

•	Based on this, we now have an accuracy of 97.89%. Since, XGBoost, being a highly optimized gradient boosting algorithm, is expected to perform very well. If it surpasses or matches RandomForest's performance, it would indicate its effectiveness for this dataset.